# SAC Training on Boeing 747 with Vectorized Environment

This notebook trains a Soft Actor-Critic (SAC) agent on the improved Boeing 747 longitudinal model using a vectorized torch environment (64 parallel envs on GPU). It uses a two-phase curriculum: tracking reward first, then step-response fine-tuning.

In [ ]:
# SAC training for B747 step-response with vectorized torch env
# Optional installs (uncomment if needed)
# %pip install tensorboard tqdm --quiet

import numpy as np
import torch

from tensoraerospace.envs.b747_vec_torch import ImprovedB747VecEnvTorch
from tensoraerospace.agent.sac.sac import SAC
from tensoraerospace.utils import convert_tp_to_sec_tp, generate_time_period

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else ("mps" if torch.backends.mps.is_available() else "cpu")
)
print("Using device:", DEVICE)

np.random.seed(1)
_ = torch.manual_seed(1)


## Vectorized Environment Setup

Create the B747 vectorized environment with step reference signals. A focused task mode narrows randomization around a 1-degree step at 5 seconds.

In [6]:
# Build vectorized environment (step-response task)
dt = 0.1
tp = generate_time_period(tn=20, dt=dt)
tps = convert_tp_to_sec_tp(tp, dt=dt)

NUM_ENVS = 64
SIGNAL_TYPE = "step"  # step reference

# Focused task: 1 degree step at 5 seconds
FOCUS_TASK_1DEG_T5 = True

# Curriculum:
#   1) train with reward_mode="tracking" (learn to follow reference + stabilize)
#   2) fine-tune with reward_mode="step_response" (reduce overshoot/settling penalties)
REWARD_MODE_PHASE1 = "tracking"
REWARD_MODE_PHASE2 = "step_response"

init_state = np.array([0.0, 0.0, 0.0, 0.0], dtype=np.float32)

if FOCUS_TASK_1DEG_T5:
    step_randomization = {
        "signal_type": SIGNAL_TYPE,
        # Very small jitter around target task (prevents anticipation + keeps diversity)
        "amplitude_deg_range": (0.95, 1.05),
        "min_abs_amplitude_deg": 0.95,
        "step_time_sec_range": (4.8, 5.2),
        "p_step": 1.0,
        "p_sine": 0.0,
    }
else:
    step_randomization = {
        "signal_type": SIGNAL_TYPE,
        # Broad randomization (robust controller)
        "amplitude_deg_range": (-10.0, 10.0),
        "min_abs_amplitude_deg": 1.0,
        "step_time_sec_range": (2.0, 10.0),
        "p_step": 1.0,
        "p_sine": 0.0,
    }

env = ImprovedB747VecEnvTorch(
    num_envs=NUM_ENVS,
    dt=dt,
    tn=float(tps[-1]),
    initial_state=init_state,
    device=DEVICE,
    seed=1,
    auto_reset=True,
    reward_mode=REWARD_MODE_PHASE1,
    step_randomization=step_randomization,
)

obs, info = env.reset()
assert obs.shape == (NUM_ENVS, 4)
print("Vec obs shape:", tuple(obs.shape))
print("num_envs:", env.num_envs)
print("device:", env.device)
print("signal_type:", SIGNAL_TYPE)
print("reward_mode_phase1:", REWARD_MODE_PHASE1)
print("reward_mode_phase2:", REWARD_MODE_PHASE2)
print("env.reward_mode (current):", env.reward_mode)
print("Action space:", env.action_space)
print("Observation space:", env.observation_space)


Vec obs shape: (64, 4)
num_envs: 64
device: cuda
signal_type: step
reward_mode_phase1: tracking
reward_mode_phase2: step_response
env.reward_mode (current): tracking
Action space: Box(-1.0, 1.0, (1,), float32)
Observation space: Box(-1.0, 1.0, (4,), float32)


## SAC Agent Configuration

Initialize the SAC agent with tuned hyperparameters, replay buffer, and TensorBoard logging.

In [7]:
# Hyperparameters (recommended for good performance on 1deg@5s)
FAST_PRESET = False

# Dedicated dirs for this run (avoid mixing old checkpoints/logs)
RUN_TAG = "sac_b747_vec64_step_1deg_t5" if bool(globals().get("FOCUS_TASK_1DEG_T5", False)) else "sac_b747_vec64_step"
log_dir = f"runs/{RUN_TAG}"
best_model_dir = f"./{RUN_TAG}_best"
save_root_dir = f"./{RUN_TAG}"

# Optional: wipe previous checkpoints for a clean run
RESET_CHECKPOINT_DIRS = True
if RESET_CHECKPOINT_DIRS:
    import shutil

    shutil.rmtree(best_model_dir, ignore_errors=True)
    shutil.rmtree(save_root_dir, ignore_errors=True)

updates_per_step = 2
batch_size = 1024
memory_capacity = 1_000_000
hidden_size = 256
warmup_steps = 20_000

# For focused single-task training it's often better to keep entropy from collapsing
# and to avoid alpha -> ~0.0.
AUTO_ENTROPY = False

agent = SAC(
    env=env,
    updates_per_step=updates_per_step,
    batch_size=batch_size,
    memory_capacity=memory_capacity,
    lr=3e-4,
    policy_lr=3e-4,
    gamma=0.99,
    tau=0.005,
    alpha=0.2,
    automatic_entropy_tuning=AUTO_ENTROPY,
    hidden_size=hidden_size,
    device=DEVICE,
    seed=336699,
    log_dir=log_dir,
    # reduce overhead: log losses every N updates (episodes/reward still logged normally)
    log_every_updates=200,
)

print("FAST_PRESET:", FAST_PRESET)
print("Policy parameters:", sum(p.numel() for p in agent.policy.parameters()))
print("Critic parameters:", sum(p.numel() for p in agent.critic.parameters()))
print("TensorBoard log_dir:", log_dir)


FAST_PRESET: False
Policy parameters: 67586
Critic parameters: 135170
TensorBoard log_dir: runs/sac_b747_vec64_step_1deg_t5


## Two-Phase Curriculum Training

Phase 1 trains with tracking reward (200k steps) to learn reference following. Phase 2 fine-tunes with step-response reward (100k steps) to minimize overshoot and settling time.

In [8]:
# Train (vectorized) — curriculum for fast convergence on 1deg@5s
# TensorBoard:
#   tensorboard --logdir runs/sac_b747_vec64_step_1deg_t5

PHASE1_STEPS = 200_000  # tracking
PHASE2_STEPS = 100_000  # step_response fine-tune

print("Phase 1:", REWARD_MODE_PHASE1, "steps=", PHASE1_STEPS)
env.reward_mode = REWARD_MODE_PHASE1
agent.train_vector(
    total_steps=PHASE1_STEPS,
    warmup_steps=warmup_steps,
    log_every=5_000,
    reward_window=200,
    save_best=False,
    save_path=best_model_dir,
    save_best_with_gradients=False,
)

print("Phase 2:", REWARD_MODE_PHASE2, "steps=", PHASE2_STEPS)
env.reward_mode = REWARD_MODE_PHASE2
agent.train_vector(
    total_steps=PHASE2_STEPS,
    warmup_steps=0,
    log_every=5_000,
    reward_window=200,
    save_best=True,
    save_path=best_model_dir,
    save_best_with_gradients=False,
)

try:
    agent.close()
except Exception:
    pass

print("Finished steps:", PHASE1_STEPS + PHASE2_STEPS)
print("TensorBoard log_dir:", log_dir)
print("Best checkpoints dir:", best_model_dir)


Phase 1: tracking steps= 200000


SAC train_vector: 100%|██████████| 200000/200000 [1:46:20<00:00, 31.35step/s, mean_R=-2.061, episodes=64448, updates=360000, replay=1e+6]  


Phase 2: step_response steps= 100000


SAC train_vector: 100%|██████████| 100000/100000 [58:26<00:00, 28.52step/s, mean_R=-16.087, episodes=32545, updates=2e+5, replay=1e+6]   

Finished steps: 300000
TensorBoard log_dir: runs/sac_b747_vec64_step_1deg_t5
Best checkpoints dir: ./sac_b747_vec64_step_1deg_t5_best
